# 对分易自动签到 APK 构建
全部运行 → 授权 Drive → 等 20-30 分钟 → APK 存到网盘。关电脑不受影响。

**如果构建失败**: 检查最后输出的错误摘要，常见问题是 Cython 版本不兼容（已修复）。

In [ ]:
# 0. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. 安装系统依赖
!sudo apt-get update -qq
!sudo apt-get install -y -qq \
    openjdk-17-jdk-headless \
    autoconf automake libtool \
    cmake zip unzip \
    libffi-dev libssl-dev \
    libxml2-dev libxslt1-dev \
    python3-pip git \
    libmtdev-dev libx11-dev libxi-dev \
    libgl1-mesa-dev libgles2-mesa-dev

In [ ]:
# 2. 安装 buildozer + cython（必须锁定版本）
!python3 -m pip install -q buildozer "cython==0.29.36" legacy-cgi

# 检查版本和 Python 3.13+ 兼容补丁
import sys, cython, buildozer, cgi
print(f"Python: {sys.version}")
print(f"Cython: {cython.__version__}")
print(f"Buildozer: {buildozer.__version__}")
print(f"cgi module: {cgi.__file__}")

In [ ]:
# 3. 克隆项目
!rm -rf duifene-sign
!git clone https://github.com/Mafeixxn/duifene-sign.git
%cd duifene-sign/android

In [ ]:
# 4. 构建 APK（约 15-30 分钟，自动重试 3 次）
import subprocess, time, os, glob, shutil

# 允许 root 用户运行 buildozer（Colab 环境必需）
os.environ['BUILDOZER_ALLOW_ROOT'] = '1'

for i in [1, 2, 3]:
    log_file = f'build_{i}.log'
    print(f'\n>>> 第 {i} 次构建（日志: {log_file}）...')
    print(f'磁盘剩余: {shutil.disk_usage(".").free // (1024**3)} GB')

    # 重试前清理构建中间文件（保留 SDK/NDK 下载）
    if i > 1:
        print('清理上次构建中间文件...')
        for pattern in [
            '.buildozer/android/platform/build-*',
            '.buildozer/android/app',
        ]:
            for p in glob.glob(pattern):
                shutil.rmtree(p, ignore_errors=True)

    with open(log_file, 'w') as f:
        r = subprocess.run(
            ['buildozer', '--verbose', 'android', 'debug'],
            env={**os.environ, 'BUILDOZER_ALLOW_ROOT': '1'},
            input=b'y\n', stdout=f, stderr=subprocess.STDOUT,
        )

    if r.returncode == 0:
        print(f'✅ 构建成功！日志: {log_file}')
        break

    print(f'❌ 失败 (exit={r.returncode})')

    if i < 3:
        print('3 秒后重试...')
        time.sleep(3)
    else:
        # 保存所有日志到 Drive 用于排查
        for lf in glob.glob('build_*.log'):
            shutil.copy(lf, f'/content/drive/MyDrive/duifene_{lf}')
        print('3 次均失败，所有 build_*.log 已存到 Drive')

In [ ]:
# 5. 显示最近的错误（帮助排查）
import glob
logs = sorted(glob.glob('build_*.log'))
if logs:
    import subprocess
    # 显示最后一个日志的关键错误
    !grep -i -E 'error:|failed:|exception|traceback|command failed' {logs[-1]} | tail -20 || echo '(无关键错误)'
    # 显示最后 30 行
    print(f'\n--- {logs[-1]} 最后 30 行 ---')
    !tail -30 {logs[-1]}

In [ ]:
# 6. 保存 APK 到 Google Drive
import glob, shutil, os

apks = glob.glob('bin/*.apk')
if apks:
    dest = '/content/drive/MyDrive/duifene_sign.apk'
    shutil.copy(apks[0], dest)
    mb = os.path.getsize(apks[0]) / (1024*1024)
    print(f'✅ APK: {apks[0]} ({mb:.1f}MB) → Drive 根目录 duifene_sign.apk')
else:
    print('❌ 未找到 APK，构建可能失败')
    print('请检查上面的 build_*.log，或 Drive 中的 duifene_build_*.log')